In [12]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForQuestionAnswering,
    pipeline
)
import torch

# ---------- Text Summarization ----------

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

article = """
Generative AI refers to a class of artificial intelligence models capable of
producing new content such as text, images, audio, and video. Large Language Models (LLMs)
such as GPT and LLaMA are trained on massive text corpora and can perform a wide range of
natural language tasks including translation, summarization, and question answering. These
models are increasingly being deployed in industry applications ranging from customer support
to software development, transforming how humans interact with machines.
"""

inputs = tokenizer(
    article,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=45,
    min_length=20,
    do_sample=False
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Summary:")
print(summary)


# ---------- Question Answering ----------

qa_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-cased-distilled-squad"
)

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    "distilbert-base-cased-distilled-squad"
)

question = "What are Large Language Models trained on?"

qa_inputs = qa_tokenizer(
    question,
    article,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = qa_model(**qa_inputs)

answer_start = torch.argmax(outputs.start_logits)
answer_end = torch.argmax(outputs.end_logits) + 1

answer = qa_tokenizer.decode(
    qa_inputs["input_ids"][0][answer_start:answer_end],
    skip_special_tokens=True
)

confidence = torch.max(outputs.start_logits).item()

print("\nQuestion:", question)
print("Answer:", answer)
print("Confidence:", round(confidence, 3))

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Summary:
Large Language Models (LLMs) are trained on massive text corpora. They can perform a wide range of natural language tasks including translation, summarization, and question answering.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


Question: What are Large Language Models trained on?
Answer: massive text corpora
Confidence: 12.206
